In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load data
df = pd.read_csv('cbc_final_data.csv')

# Filter Females and Drop unnecessary columns
df_female = df[df['Gender'] == 0].copy()
df_female = df_female.drop(['Gender', 'Diagnosis'], axis=1, errors='ignore')

if 'weight_kg' in df_female.columns:
    df_female = df_female.drop('weight_kg', axis=1)

print(f"Data ready. Records: {len(df_female)}")

Data ready. Records: 3426


In [2]:
labels = [
    'Leukopenia', 'Leukocytosis', 'Anemia', 'Elevated Hemoglobin',
    'Thrombocytopenia', 'Thrombocytosis', 'Microcytosis', 'Macrocytosis',
    'Elevated RDW', 'Neutropenia', 'Neutrophilia', 'Lymphopenia', 'Lymphocytosis',
    'Monocytopenia', 'Monocytosis', 'Eosinophilia', 'Basophilia',
    'Probable Iron Deficiency Anemia (IDA)'
]

def apply_rules(row):
    d = []
    if row['WBC'] < 4.0: d.append('Leukopenia')
    if row['WBC'] > 10.0: d.append('Leukocytosis')
    if row['Hb'] < 12.0: d.append('Anemia')
    if row['Hb'] > 15.5: d.append('Elevated Hemoglobin')
    if row['PLATELETS'] < 150: d.append('Thrombocytopenia')
    if row['PLATELETS'] > 450: d.append('Thrombocytosis')
    if row['MCV'] < 80: d.append('Microcytosis')
    if row['MCV'] > 100: d.append('Macrocytosis')
    if row['RDW'] > 15.0: d.append('Elevated RDW')
    if row['MCV'] < 80.0 and row['RDW'] > 15.0:
        d.append('Probable Iron Deficiency Anemia (IDA)')
    return d

y_list = df_female.apply(apply_rules, axis=1)
mlb = MultiLabelBinarizer(classes=labels)
y = mlb.fit_transform(y_list)

In [3]:
features = ['Age', 'Hb', 'RBC', 'WBC', 'PLATELETS', 'HCT', 'MCV', 'MCH', 'MCHC',
            'RDW', 'PDW', 'MPV', 'PCT', 'Lymphocytes', 'Monocytes', 'Neutrophils',
            'Eosinophils', 'Basophils']

X = df_female[features]

# Split data before scaling to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# Train Multi-label Random Forest
rf_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
rf_model.fit(X_train_scaled, y_train)

# Predict and Calculate Accuracy
y_pred = rf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Subset Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=labels))

Subset Accuracy: 99.85%

Detailed Classification Report:
                                       precision    recall  f1-score   support

                           Leukopenia       1.00      1.00      1.00        37
                         Leukocytosis       1.00      1.00      1.00       387
                               Anemia       1.00      1.00      1.00       351
                  Elevated Hemoglobin       1.00      1.00      1.00        73
                     Thrombocytopenia       1.00      0.99      0.99        77
                       Thrombocytosis       1.00      1.00      1.00        78
                         Microcytosis       1.00      1.00      1.00       336
                         Macrocytosis       1.00      1.00      1.00         4
                         Elevated RDW       1.00      1.00      1.00       187
                          Neutropenia       0.00      0.00      0.00         0
                         Neutrophilia       0.00      0.00      0.00     

C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitali

In [5]:
# Save model and preprocessing objects
joblib.dump(rf_model, 'female_cbc_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(mlb, 'mlb.pkl')
joblib.dump(features, 'features_list.pkl')

print("All files saved successfully!")

All files saved successfully!
